# 퍼널 분석용 전처리

**입력**: `funnel_master_df.csv` (1주문=1행, 11컬럼)
**출력**: `funnel_df.csv` (구간별 소요시간 + 유효 주문 플래그 추가)

**핵심 질문**: 주문 완료 → 결제 승인 → 물류 인도 → 배송 완료 의 4개 시각 사이에서 **어느 구간이 가장 오래 걸리는가?** (= "판매자 vs 택배사 누구 탓")

**전처리 내용**
1. timestamp 컬럼 파싱
2. 구간별 소요시간 계산 (days)
3. 약속 vs 실제 (`delay_days`, `is_delayed`)
4. 분석에 사용할 수 있는 유효 주문 플래그 (`is_valid_funnel`)

In [1]:
import pandas as pd
import numpy as np

# 상대경로 — 같은 폴더의 funnel_master_df.csv 로드
# csv 로 저장된 timestamp는 문자열이므로 parse_dates 로 다시 datetime 복원
DATE_COLS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

funnel_df = pd.read_csv("funnel_master_df.csv", parse_dates=DATE_COLS)
print("shape:", funnel_df.shape)
print("dtypes (timestamp 4개 확인):")
print(funnel_df[DATE_COLS].dtypes)


shape: (99441, 11)
dtypes (timestamp 4개 확인):
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [2]:
# === 1) 구간별 소요시간 (단위: days, float) ==================================
# 주문 완료 → 결제 승인 → 물류 인도 → 배송 완료
def _to_days(td):
    """Timedelta Series → days(float). NaT 은 NaN."""
    return td.dt.total_seconds() / 86400.0

# 구간 1: 결제 승인까지 (주문 → 승인)
funnel_df["t_approve_d"] = _to_days(
    funnel_df["order_approved_at"] - funnel_df["order_purchase_timestamp"]
)

# 구간 2: 판매자가 택배사에 넘기기까지 (승인 → 택배사 인도)  ← "판매자 책임 구간"
funnel_df["t_carrier_d"] = _to_days(
    funnel_df["order_delivered_carrier_date"] - funnel_df["order_approved_at"]
)

# 구간 3: 택배사가 고객에게 배달까지 (택배사 인도 → 배송 완료)  ← "택배사 책임 구간"
funnel_df["t_delivery_d"] = _to_days(
    funnel_df["order_delivered_customer_date"] - funnel_df["order_delivered_carrier_date"]
)

# 전체 리드타임 (주문 → 배송 완료)
funnel_df["t_total_d"] = _to_days(
    funnel_df["order_delivered_customer_date"] - funnel_df["order_purchase_timestamp"]
)

print(funnel_df[["t_approve_d", "t_carrier_d", "t_delivery_d", "t_total_d"]].describe().round(2))


       t_approve_d  t_carrier_d  t_delivery_d  t_total_d
count     99281.00     97644.00      96475.00   96476.00
mean          0.43         2.81          9.33      12.56
std           1.08         3.55          8.76       9.55
min           0.00      -171.22        -16.10       0.53
25%           0.01         0.88          4.10       6.77
50%           0.01         1.82          7.10      10.22
75%           0.61         3.58         12.03      15.72
max         187.88       125.76        205.19     209.63


In [3]:
# === 2) 약속 vs 실제 — 지연 여부 =============================================
# Olist 가 고객에게 약속한 예상 배송일 대비 실제 배송이 얼마나 늦/빠른지
funnel_df["delay_days"] = _to_days(
    funnel_df["order_delivered_customer_date"] - funnel_df["order_estimated_delivery_date"]
)
# True 면 약속보다 늦게 도착. 배송 미완(NaT) 은 NaN 유지 → 비교 대상에서 자동 제외
funnel_df["is_delayed"] = (funnel_df["delay_days"] > 0).where(
    funnel_df["order_delivered_customer_date"].notna(), other=np.nan
)

print("배송 완료 주문 중 지연 비율:", round(funnel_df["is_delayed"].mean(), 4))
print(funnel_df["is_delayed"].value_counts(dropna=False))


배송 완료 주문 중 지연 비율: 0.0811
is_delayed
False    88649
True      7827
NaN       2965
Name: count, dtype: int64


In [4]:
# === 3) 분석에 사용할 수 있는 "유효 주문" 플래그 ==============================
# 퍼널 분석은 4개 timestamp 가 모두 있고, 시간 순서도 맞는 주문만 의미가 있음.
# (일부 주문은 timestamp 가 어긋나 음수 duration 이 발생 — 데이터 품질 이슈)

# 조건 ① 4개 timestamp 모두 존재 (= 배송 완료)
has_all_ts = funnel_df[
    ["order_purchase_timestamp", "order_approved_at",
     "order_delivered_carrier_date", "order_delivered_customer_date"]
].notna().all(axis=1)

# 조건 ② 모든 구간 소요시간이 음수가 아님 (시간 순서가 맞음)
valid_durations = (
    (funnel_df["t_approve_d"]  >= 0) &
    (funnel_df["t_carrier_d"]  >= 0) &
    (funnel_df["t_delivery_d"] >= 0)
)

funnel_df["is_valid_funnel"] = has_all_ts & valid_durations

print("전체:", len(funnel_df))
print("4개 timestamp 모두 있음:", has_all_ts.sum())
print("음수 duration 없음     :", valid_durations.sum())
print("최종 is_valid_funnel    :", funnel_df["is_valid_funnel"].sum(),
      f"({funnel_df['is_valid_funnel'].mean()*100:.2f}%)")


전체: 99441
4개 timestamp 모두 있음: 96461
음수 duration 없음     : 95088
최종 is_valid_funnel    : 95088 (95.62%)


In [5]:
# === 4) 유효 주문만으로 구간별 중앙값 미리보기 (분석 결과 미리 확인) ===========
valid = funnel_df[funnel_df["is_valid_funnel"]]
print("유효 주문:", len(valid))
print()
print("구간별 중앙값 (days):")
print(valid[["t_approve_d", "t_carrier_d", "t_delivery_d", "t_total_d"]].median().round(2))
print()
print("→ 가장 오래 걸린 구간:",
      valid[["t_approve_d", "t_carrier_d", "t_delivery_d"]].median().idxmax())


유효 주문: 95088

구간별 중앙값 (days):
t_approve_d      0.01
t_carrier_d      1.85
t_delivery_d     7.11
t_total_d       10.27
dtype: float64

→ 가장 오래 걸린 구간: t_delivery_d


In [6]:
# === 5) 저장 =================================================================
funnel_df.to_csv("funnel_df.csv", index=False)

print("최종 shape:", funnel_df.shape)
print("저장 경로 : funnel_df.csv")
print()
print("컬럼 목록:")
for c in funnel_df.columns:
    print(" -", c)


최종 shape: (99441, 18)
저장 경로 : funnel_df.csv

컬럼 목록:
 - order_id
 - customer_id
 - order_status
 - order_purchase_timestamp
 - order_approved_at
 - order_delivered_carrier_date
 - order_delivered_customer_date
 - order_estimated_delivery_date
 - customer_unique_id
 - customer_state
 - review_score
 - t_approve_d
 - t_carrier_d
 - t_delivery_d
 - t_total_d
 - delay_days
 - is_delayed
 - is_valid_funnel


---

# 📒 전처리 정리

## 1. 무엇을 만들었나
**`funnel_df.csv`** = `funnel_master_df.csv` + 분석에 필요한 파생 컬럼 6개

| 추가된 컬럼 | 의미 | 단위 |
|---|---|---|
| `t_approve_d`     | 주문 → 결제 승인 | days |
| `t_carrier_d`     | 결제 승인 → 택배사 인도 (**판매자 책임**) | days |
| `t_delivery_d`    | 택배사 인도 → 배송 완료 (**택배사 책임**) | days |
| `t_total_d`       | 주문 → 배송 완료 (전체 리드타임) | days |
| `delay_days`      | 실제 배송 − 예상 배송 | days |
| `is_delayed`      | 약속보다 늦었는지 | bool |
| `is_valid_funnel` | 4 단계 모두 채워졌고 음수 구간 없음 | bool |

행 수는 그대로 99,441 (1주문=1행).

## 2. 단계별 처리

### ① 데이터 로드
- csv 의 timestamp 는 문자열이라 **`parse_dates=` 필수**.
- 안 그러면 `df["a"] - df["b"]` 가 에러 나거나 의미 없는 값이 됨.

### ② 구간별 소요시간 (Timedelta → days, float)
```python
def _to_days(td):
    return td.dt.total_seconds() / 86400.0
```
- `td.dt.days` 를 안 쓰는 이유: 정수로 잘림 (12시간 = 0일 됨).
- `total_seconds() / 86400` 으로 **0.5일 같은 소수점 보존**.

### ③ 약속 vs 실제 (`is_delayed`)
- `delay_days > 0` 면 약속보다 늦은 것.
- 배송이 아직 안 된 주문(NaT)은 `is_delayed` 를 NaN 으로 둠 → 평균·집계에서 자동 제외됨.
- `.where(notna(), other=np.nan)` 패턴: bool Series 를 NaN 허용하려면 object/float 로 만들어야 해서 이 트릭이 필요.

### ④ 유효 주문 플래그 (`is_valid_funnel`)
퍼널 분석은 두 조건을 모두 만족해야 의미가 있음:

1. **4개 timestamp 모두 존재** → 사실상 `order_status == "delivered"` 와 같음
2. **모든 구간이 음수가 아님** → 일부 주문은 timestamp 가 어긋남 (예: 승인 시각이 주문 시각보다 빠른 경우). 데이터 품질 이슈 → 음수 구간이 있는 행은 제외.

**행을 직접 잘라내지 않은 이유**: 분석 노트북에서 `df[df["is_valid_funnel"]]` 로 필요할 때만 필터하고, 원본 결측·이상치는 그대로 남겨 진단할 수 있게 하기 위함.

### ⑤ 중앙값 미리보기
저장 전에 `t_carrier_d` vs `t_delivery_d` 중앙값을 비교 → **"가장 오래 걸리는 구간"** 을 미리 출력해서 핵심 질문에 즉답.

## 3. 분석 노트북에서 쓰는 법

```python
import pandas as pd

DATE_COLS = [...]   # 위 전처리 셀과 동일
df = pd.read_csv("funnel_df.csv", parse_dates=DATE_COLS)

# 유효 주문만 사용
valid = df[df["is_valid_funnel"]].copy()

# 히스토그램 / 박스플롯
import seaborn as sns
stage_cols = ["t_approve_d", "t_carrier_d", "t_delivery_d"]
long = valid[stage_cols].melt(var_name="stage", value_name="days")
sns.boxplot(data=long, x="stage", y="days")
```

## 4. 주의
- `funnel_df.csv` 다시 읽을 때도 **`parse_dates=DATE_COLS` 꼭 지정**.
- 평균보다는 **중앙값**을 보는 게 좋음 — 일부 주문은 100일 넘는 outlier 가 있어 평균이 왜곡됨.
- 박스플롯에서 outlier 가 너무 많으면 `showfliers=False` 또는 `y` 축에 상한 (예: 30일) 을 걸면 분포 비교가 깔끔해짐.
- "판매자 vs 택배사" 결론을 낼 때 평균/중앙값뿐 아니라 **분포 형태(skew, 긴 꼬리)** 도 같이 봐야 함 — 같은 중앙값이어도 꼬리가 두꺼우면 변동성이 큰 쪽이 진짜 병목.